Systematic Model Comparison Pipeline

Compare multiple models using same CV folds with sklearn Pipeline

In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
import kagglehub
from kagglehub import KaggleDatasetAdapter, dataset_load
import time

# Prepare data
file_path = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "blastchar/telco-customer-churn",
  file_path,
)

# Convert 'TotalCharges' to numeric, handling missing/non-numeric values
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(0, inplace=True) # Fill NaNs created by coercion with 0

target = 'Partner'
# Convert target variable to numerical (0 or 1)
y = df[target].map({'Yes': 1, 'No': 0})

# Drop 'customerID' and the original 'target' column from features
X = df.drop(columns=['customerID', target])

# Identify categorical columns for one-hot encoding
categorical_cols = X.select_dtypes(include='object').columns
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True) # One-hot encode categorical features

X_train , X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, stratify=y, random_state=42
)

models = {
    '1. Baseline': DummyClassifier(strategy='most_frequent'),
    '2. LogRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000))
    ]),
    '3. Radom Forest': RandomForestClassifier(
        n_estimators=100, random_state=42
    ),
    '4. Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, random_state=42
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results =[]

for name, model in models.items():
  start = time.time()
  scores = cross_val_score(
      model, X_train, y_train, cv=cv, scoring='accuracy'
  )
  elapsed = time.time() - start
  results.append({
      'Model:' : name,
      'CV Mean' : f"{scores.mean():.3f}",
      "CV Std" : f"{scores.std():.3f}",
      "Time" : f"{elapsed:.2f}"
  })
  print(f"{name}: {scores.mean():.3f} +/- {scores.std():.3f}")

#Evaluation
df_results = pd.DataFrame(results) # Fixed typo: pd.Dataframe -> pd.DataFrame
print("\n" + df_results.to_string(index=False))

best = models['4. Gradient Boosting']
best.fit(X_train, y_train)
print(f"\nTest Score: {best.score(X_test, y_test):.3f}")

Using Colab cache for faster access to the 'telco-customer-churn' dataset.
1. Baseline: 0.517 +/- 0.000


/tmp/ipykernel_911/3452578206.py:24: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(0, inplace=True) # Fill NaNs created by coercion with 0


2. LogRegression: 0.754 +/- 0.005
3. Radom Forest: 0.738 +/- 0.008
4. Gradient Boosting: 0.755 +/- 0.004

              Model: CV Mean CV Std Time
         1. Baseline   0.517  0.000 0.02
    2. LogRegression   0.754  0.005 0.27
     3. Radom Forest   0.738  0.008 4.91
4. Gradient Boosting   0.755  0.004 6.19

Test Score: 0.742
